# ============================================================
# Samskara Public Health Ingestion - Phase 1
# API -> Remote Raw Storage -> Bronze -> Audit -> Checkpoint
# ============================================================

In [ ]:
import os
import json
import uuid
import requests
import pandas as pd
from datetime import datetime, timezone
from pathlib import Path

# ----------------------------
# 1. Config
# ----------------------------

In [ ]:
RUN_ID = str(uuid.uuid4())
RUN_TS = datetime.now(timezone.utc)
RUN_DATE = RUN_TS.strftime("%Y-%m-%d")
RUN_TS_STR = RUN_TS.strftime("%Y%m%d_%H%M%S")

# raw_path    = sm.storage_path("arrowlake-S3", "public_health", "raw")
# bronze_path = sm.storage_path("arrowlake-S3", "public_health", "bronze")
# → "s3://my-bucket/public_health/raw"
# → "s3://my-bucket/public_health/bronze"

DATASET_NAME = "cdc_public_health"
SOURCE_SYSTEM = "cdc_socrata"

# Example CDC Socrata dataset endpoint
# Replace dataset_id later with the final dataset you choose
CDC_DATASET_ID = "9bhg-hcku"
API_URL = f"https://data.cdc.gov/resource/{CDC_DATASET_ID}.json"

LIMIT_ROWS = 5000
max_rows = sm.param("MAX_ROWS", 5000)
if not max_rows:
    max_rows = LIMIT_ROWS
    print("not yet populated by scheduler")
else:
    print("populated by scheduler")

# RAW_PATH = "/public_health/raw"
# REMOTE_ARROWLAKE_DB = "s3://YOUR_BUCKET/public_health/arrowlake/public_health_arrowlake"

# LOCAL_TMP_DIR = "/tmp/samskara_public_health"
# Path(LOCAL_TMP_DIR).mkdir(parents=True, exist_ok=True)

RAW_FILE_NAME = f"{DATASET_NAME}_{RUN_TS_STR}_{RUN_ID}.json"
# LOCAL_RAW_FILE = f"{LOCAL_TMP_DIR}/{RAW_FILE_NAME}"

RAW_REMOTE_PATH = (
    f"{raw_path}/dataset={DATASET_NAME}/"
    f"ingestion_date={RUN_DATE}/run_id={RUN_ID}/"
)

RAW_REMOTE_PATH_WITH_FILE_NAME = (
    f"{RAW_REMOTE_PATH}/{RAW_FILE_NAME}"
)

print("RUN_ID:", RUN_ID)
print("API_URL:", API_URL)
print("RAW_REMOTE_PATH:", RAW_REMOTE_PATH)
print("RAW_REMOTE_PATH_WITH_FILE_NAME:", RAW_REMOTE_PATH_WITH_FILE_NAME)
print(max_rows)

# ----------------------------
# 2. Fetch source data
# ----------------------------

In [ ]:
params = {
    "$limit": max_rows
}

response = requests.get(API_URL, params=params, timeout=60)
response.raise_for_status()

records = response.json()

# Path(RAW_REMOTE_PATH).mkdir(parents=True, exist_ok=True)

# with open(RAW_REMOTE_PATH_WITH_FILE_NAME, "w", encoding="utf-8") as f:
#     json.dump(records, f, ensure_ascii=False, indent=2)

# Replace Path().mkdir() + open() with:
with sm.open_file("arrowlake-S3", RAW_REMOTE_PATH_WITH_FILE_NAME, "w") as f:
    json.dump(records, f, ensure_ascii=False, indent=2)

row_count = len(records)

print(f"Fetched rows: {row_count}")
print(f"Remote raw file: {RAW_REMOTE_PATH}")

# ----------------------------
# 3. Upload raw file to remote storage
# Requires fsspec + s3fs configured in Samskara environment
# ----------------------------

# ----------------------------
# 4. Convert to dataframe
# ----------------------------

In [ ]:
df = pl.DataFrame(records)

if df.is_empty():
    print("No data received from source.")
else:
    # Polars requires using .with_columns() to add new columns
    df = df.with_columns(
        _source_system = pl.lit(SOURCE_SYSTEM),
        _dataset_name = pl.lit(DATASET_NAME),
        _dataset_id = pl.lit(CDC_DATASET_ID),
        _run_id = pl.lit(RUN_ID),
        _raw_file_path = pl.lit(RAW_REMOTE_PATH),
        _ingestion_ts_utc = pl.lit(RUN_TS.isoformat()),
        _ingestion_date = pl.lit(RUN_DATE)
    )

# ----------------------------
# 5. Connect to ArrowLake database
# Adjust this depending on your Samskara ArrowLake connection name
# ----------------------------

In [ ]:
try:
  started_at = RUN_TS

  if df.is_empty():
    bronze_count = 0
  else:
    bronze_rows = [
        {
            "run_id": RUN_ID,
            "dataset_name": DATASET_NAME,
            "dataset_id": CDC_DATASET_ID,
            "source_system": SOURCE_SYSTEM,
            "raw_file_path": RAW_REMOTE_PATH,
            "raw_record_json": json.dumps(r),
            "ingestion_ts_utc": RUN_TS,
            "ingestion_date": RUN_DATE,
        }
        for r in records
    ]

    bronze_df = pd.DataFrame(bronze_rows)

    bronze_result = sm.write_arrowdelta(
    bronze_df,
    "bronze.public_health_events",
    mode="append", partition_by="ingestion_date")

    bronze_count = sm.sql_arrowdelta(f"SELECT COUNT(*) FROM public_health.bronze.public_health_events WHERE run_id = '{RUN_ID}'").iloc[0, 0]

    completed_at = datetime.now(timezone.utc)

    etl_audit_row = [
        RUN_ID,
        DATASET_NAME,
        SOURCE_SYSTEM,
        CDC_DATASET_ID,
        API_URL,
        RAW_REMOTE_PATH,
        row_count,
        bronze_count,
        "SUCCESS",
        None,
        started_at,
        completed_at,
        RUN_DATE
    ]
    
    etl_audit_df = pd.DataFrame([etl_audit_row], columns=[
    "run_id", "dataset_name", "source_system", "cdc_dataset_id",
    "api_url", "raw_remote_path", "row_count", "error_count",
    "status", "error_msg", "run_ts", "completed_at", "run_date"
    ])

    etl_audit_result = sm.write_arrowdelta(
    etl_audit_df,
    "etl.etl_ingestion_audit",
    mode="append", partition_by="run_date")

    if sm.table_exists_arrowdelta("public_health.etl.etl_file_checkpoint"):
        sm.sql_arrowdelta(f"DELETE FROM public_health.etl.etl_file_checkpoint WHERE dataset_name = '{DATASET_NAME}' AND cdc_dataset_id = '{CDC_DATASET_ID}'")

    etl_cp_row = [
        DATASET_NAME,
        CDC_DATASET_ID,
        RUN_ID,
        RAW_REMOTE_PATH,
        completed_at,
        row_count,
        "SUCCESS"
    ]

    etl_cp_df = pd.DataFrame([etl_cp_row], columns=[
    "dataset_name", "cdc_dataset_id",
    "run_id", "raw_remote_path", "completed_at", "row_count", "run_date"
    ])

    etl_cp_result = sm.write_arrowdelta(
    etl_cp_df,
    "etl.etl_file_checkpoint",
    mode="append", partition_by="run_date")

    print("Bronze load successful")
    print("Bronze rows inserted:", bronze_count)

except Exception as e:
    completed_at = datetime.now(timezone.utc)

    etl_audit_row = [
    RUN_ID,
    DATASET_NAME,
    SOURCE_SYSTEM,
    CDC_DATASET_ID,
    API_URL,
    RAW_REMOTE_PATH,
    row_count,
    0,
    "FAILED",
    str(e),
    RUN_TS,
    completed_at,
    RUN_DATE
    ]

    etl_audit_df = pd.DataFrame([etl_audit_row], columns=[
    "run_id", "dataset_name", "source_system", "cdc_dataset_id",
    "api_url", "raw_remote_path", "row_count", "error_count",
    "status", "error_msg", "run_ts", "completed_at", "run_date"
    ])

    etl_audit_result = sm.write_arrowdelta(
    etl_audit_df,
    "etl.etl_ingestion_audit",
    mode="append", partition_by="run_date")

    raise